In [109]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.utils.parametrize as parametrize
import torchvision.datasets as datasets
import torchvision.transforms as transforms

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# Singular Value Decomposition (SVD)

In [110]:
# Generate a rank-deficient matrix W

d, k = 10, 10

W_rank = 2
W = torch.randn(d, W_rank) @ torch.randn(W_rank, k) # W shape: (10, 2) @ (2, 10) = (10, 10)
W.shape # shape: (10, 10)

torch.Size([10, 10])

In [111]:
# Evaluate the rank of matrix W
W_rank = np.linalg.matrix_rank(W)

# The rank is the number of linearly independent rows or columns in a matrix
# The rank is 2 because, a linear transformation was used to make the 10x10 matrix
print(f"Rank of matrix W: {W_rank}") # 2

Rank of matrix W: 2


In [112]:
# Compute the Singular Value Decomposition (SVD) of matrix W (W = U x S x V^T)
U, S, V = torch.svd(W)

U.shape, S.shape, V.shape

(torch.Size([10, 10]), torch.Size([10]), torch.Size([10, 10]))

In [113]:
U @ torch.diag(S) @ V

tensor([[ 2.4671e+00, -1.0677e+00,  1.6069e+00, -5.3706e-01, -3.5900e-01,
         -6.1058e-01,  5.5727e+00,  1.1296e+00,  2.8930e+00,  6.1081e-01],
        [-2.8342e+00,  2.5122e-01,  2.1358e+00,  1.9595e-01, -1.3458e+00,
          1.2030e+00, -3.1334e+00,  1.5273e+00,  3.4673e-01, -2.0976e+00],
        [ 1.3976e+00, -2.5563e-02, -1.4546e+00, -5.4187e-02,  8.4088e-01,
         -6.4378e-01,  1.2157e+00, -1.0379e+00, -5.4097e-01,  1.1751e+00],
        [ 2.5380e+00, -7.2835e-01,  1.4248e-01, -3.9277e-01,  2.9769e-01,
         -8.1840e-01,  4.4929e+00,  9.0326e-02,  1.5838e+00,  1.1579e+00],
        [-9.1297e-01,  6.0345e-02,  7.7202e-01,  5.4238e-02, -4.7060e-01,
          3.9809e-01, -9.4038e-01,  5.5159e-01,  1.8913e-01, -7.0515e-01],
        [ 2.5428e-01,  1.0666e-01, -7.1907e-01,  3.8190e-02,  3.5364e-01,
         -1.7437e-01, -1.5184e-01, -5.1123e-01, -5.1728e-01,  3.7310e-01],
        [ 1.7422e-02,  6.9259e-02, -3.0219e-01,  2.9360e-02,  1.3591e-01,
         -4.3804e-02, -2.1802e-0

In [114]:
# For rank-r factorization, only keep the first r singular values and the corresponding columns of U and V
U_r = U[:, :W_rank] # shape: (10, 2)
S_r = torch.diag(S[:W_rank]) # shape: (2, 2)
V_r = V[:, :W_rank].t() # V_r needs to be transposed to get the right dimensions, transposed shape: (2, 10)

# Compute C = U_r @ S_r and R = V_r
B = U_r @ S_r
A = V_r

print(f"Shape of B: {B.shape}")
print(f"Shape of A: {A.shape}")

Shape of B: torch.Size([10, 2])
Shape of A: torch.Size([2, 10])


In [115]:
# Given the same input, check the output using the original W matrix and the matrices resulting from the decomposition

b = torch.randn(d)
x = torch.randn(d)

# Compute y = Wx + b
y = W @ x + b

# Compute y' = (B @ A)x + b
y_prime = (B @ A) @ x + b

print(f"Total parameters of W: {W.numel()}")
print(f"y originally computed using W:\n{y}\n\n")

print(f"Total parameters of B and A: {B.numel() + A.numel()}")
print(f"y' computed using BA:\n{y_prime}")

Total parameters of W: 100
y originally computed using W:
tensor([ 8.7740, -3.2261,  0.4063,  7.2494, -0.3587, -0.8469, -1.2226,  1.8885,
         0.1394,  0.2454])


Total parameters of B and A: 40
y' computed using BA:
tensor([ 8.7740, -3.2261,  0.4063,  7.2494, -0.3587, -0.8469, -1.2226,  1.8885,
         0.1394,  0.2454])


# Low-Rank Adaptation (LoRA)

In [116]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load the MNIST train dataset
train_dataset = datasets.MNIST(root="./data",
                               train=True,
                               download=True,
                               transform=transform)

train_dataloader = torch.utils.data.DataLoader(train_dataset,
                                               batch_size=10,
                                               shuffle=True)

# Load the MNIST test dataset
test_dataset = datasets.MNIST(root="./data",
                              train=False,
                              download=True,
                              transform=transform)

test_dataloader = torch.utils.data.DataLoader(test_dataset,
                                              batch_size=10,
                                              shuffle=True)

len(train_dataloader), len(test_dataloader)

(6000, 1000)

In [117]:
class BaseNetwork(nn.Module):
    def __init__(self,
                 in_features: int = 28 * 28,
                 hidden_features: int = 1000,
                 out_features: int = 10):
        super().__init__()
        self.linear_1 = nn.Linear(in_features=in_features, out_features=hidden_features)
        self.linear_2 = nn.Linear(in_features=hidden_features, out_features=hidden_features * 2)
        self.linear_3 = nn.Linear(in_features=hidden_features * 2, out_features=out_features)

        self.relu = nn.ReLU()

    def forward(self, img):
        x = img.view(-1, 28 * 28)
        x = self.relu(self.linear_1(x))
        x = self.relu(self.linear_2(x))
        x = self.linear_3(x)

        return x

base_model = BaseNetwork(in_features=28 * 28, hidden_features=1000, out_features=10).to(device)

In [118]:
# Train the network for a single epoch to simulate pre-training

def train(train_dataloader: torch.utils.data.DataLoader,
          model: nn.Module,
          epochs: int = 5):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()

        train_loss = 0

        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")
        for (x, y) in progress_bar:
            optimizer.zero_grad()
            x, y = x.to(device), y.to(device)

            y_pred = model(x)

            loss = loss_fn(y_pred, y)
            train_loss += loss.item()

            progress_bar.set_postfix(loss=train_loss / len(train_dataloader))

            loss.backward()
            optimizer.step()

train(train_dataloader=train_dataloader, model=base_model, epochs=1)

Epoch 1: 100%|██████████| 6000/6000 [00:28<00:00, 208.25it/s, loss=0.237]


In [119]:
def test(test_dataloader: torch.utils.data.DataLoader,
         model: nn.Module):
    correct = 0
    total = 0

    wrong_counts = [0 for i in range(10)]

    for (x, y) in tqdm(test_dataloader, desc="Testing"):
        x, y = x.to(device), y.to(device)

        with torch.inference_mode():
            y_pred = model(x)

        for idx, i in enumerate(y_pred):
            if torch.argmax(i) == y[idx]:
                correct +=1
            else:
                wrong_counts[y[idx]] += 1

            total += 1

    print(f"\nAccuracy: {round(correct / total, 3)}\n")

    for i in range(len(wrong_counts)):
        print(f"Wrong counts for the digit {i}: {wrong_counts[i]}")

test(test_dataloader=test_dataloader, model=base_model)

Testing: 100%|██████████| 1000/1000 [00:03<00:00, 325.04it/s]


Accuracy: 0.965

Wrong counts for the digit 0: 24
Wrong counts for the digit 1: 15
Wrong counts for the digit 2: 27
Wrong counts for the digit 3: 31
Wrong counts for the digit 4: 36
Wrong counts for the digit 5: 37
Wrong counts for the digit 6: 34
Wrong counts for the digit 7: 46
Wrong counts for the digit 8: 40
Wrong counts for the digit 9: 63


In [120]:
original_parameters = {}
num_original_parameters = 0

for name, param in base_model.named_parameters():
    original_parameters[name] = param.clone().detach()
    num_original_parameters += param.numel()

print(f"Original model has {num_original_parameters:,} parameters")

Original model has 2,807,010 parameters


In [121]:
class LoRAParameterization(nn.Module):
    def __init__(self,
                 in_features: int,
                 out_features: int,
                 rank: int = 1,
                 alpha: int = 2,
                 device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")):
        super().__init__()

        # Use a random Gaussian initialization for A and zero for B, so ∆W = BA is zero at the beginning of training
        self.lora_A = nn.Parameter(torch.zeros((rank, out_features)).to(device)) # shape: (rank, out_features)
        self.lora_B = nn.Parameter(torch.zeros((in_features, rank)).to(device)) # shape: (in_features, rank)

        # B @ A (∆W) shape: (in_features, out_features) - same shape as weight matrix W

        nn.init.normal_(self.lora_A, mean=0, std=1)

        # Scale ∆W by alpha / rank, where alpha is a constant in rank (usually alpha = 2 * rank)
        self.scale = alpha / rank
        self.enabled = True

    def forward(self, W: torch.Tensor):
        if self.enabled:
            # W + (B @ A) * scale
            return W + torch.matmul(self.lora_B, self.lora_A) * self.scale
        else:
            return W

In [122]:
def linear_layer_lora_parameterization(layer: nn.Linear,
                                       rank: int = 1,
                                       alpha: int = 2,
                                       device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")):
    # Only add the LoRA parameterization to the weight matrix W, not the bias

    in_features, out_features = layer.weight.shape
    return LoRAParameterization(in_features=in_features,
                                out_features=out_features,
                                rank=rank,
                                alpha=alpha,
                                device=device)

parametrize.register_parametrization(base_model.linear_1,
                                     "weight",
                                     linear_layer_lora_parameterization(base_model.linear_1, rank=1, alpha=2, device=device))

parametrize.register_parametrization(base_model.linear_2,
                                     "weight",
                                     linear_layer_lora_parameterization(base_model.linear_2, rank=1, alpha=2, device=device))

parametrize.register_parametrization(base_model.linear_3,
                                     "weight",
                                     linear_layer_lora_parameterization(base_model.linear_3, rank=1, alpha=2, device=device))

def enable_disable_lora(enabled: bool = True):
    for layer in [base_model.linear_1, base_model.linear_2, base_model.linear_3]:
        layer.parametrizations["weight"][0].enabled = enabled

In [123]:
# See the number of parameters added by LoRA
num_lora_parameters = 0
num_non_lora_parameters = 0

for index, layer in enumerate([base_model.linear_1, base_model.linear_2, base_model.linear_3]):
    num_lora_parameters += layer.parametrizations["weight"][0].lora_A.numel() + layer.parametrizations["weight"][0].lora_B.numel()
    num_non_lora_parameters += layer.weight.numel() + layer.bias.numel()
    print(
        f"Layer {index+1} -\nW: {layer.weight.shape}\n+ B: {layer.bias.shape}\n+ Lora_A: {layer.parametrizations["weight"][0].lora_A.shape}\n+ Lora_B: {layer.parametrizations["weight"][0].lora_B.shape}\n\n"
    )

print(f"The model has {num_lora_parameters:,} LoRA parameters")
print(f"The model has {num_non_lora_parameters:,} non-LoRA parameters")
print(f"The model has {num_lora_parameters + num_non_lora_parameters:,} total parameters")

assert num_non_lora_parameters == num_original_parameters, "The non-LoRA parameter count must match the original model"

Layer 1 -
W: torch.Size([1000, 784])
+ B: torch.Size([1000])
+ Lora_A: torch.Size([1, 784])
+ Lora_B: torch.Size([1000, 1])


Layer 2 -
W: torch.Size([2000, 1000])
+ B: torch.Size([2000])
+ Lora_A: torch.Size([1, 1000])
+ Lora_B: torch.Size([2000, 1])


Layer 3 -
W: torch.Size([10, 2000])
+ B: torch.Size([10])
+ Lora_A: torch.Size([1, 2000])
+ Lora_B: torch.Size([10, 1])


The model has 6,794 LoRA parameters
The model has 2,807,010 non-LoRA parameters
The model has 2,813,804 total parameters


In [124]:
# Freeze the non-Lora parameters
for name, param in base_model.named_parameters():
    if "lora" not in name:
        param.requires_grad = False

In [125]:
# Load the MNIST train dataset, only keeping the digit 0
train_9_dataset = datasets.MNIST(root="./data",
                               train=True,
                               download=True,
                               transform=transform)

exclude_indices = train_9_dataset.targets == 9
train_9_dataset.data = train_9_dataset.data[exclude_indices]
train_9_dataset.targets = train_9_dataset.targets[exclude_indices]

train_9_dataloader = torch.utils.data.DataLoader(train_9_dataset,
                                                 batch_size=10,
                                                 shuffle=True)

len(train_9_dataloader)

595

In [126]:
train(train_dataloader=train_9_dataloader, model=base_model, epochs=1)

Epoch 1: 100%|██████████| 595/595 [00:03<00:00, 185.39it/s, loss=0.015]


In [127]:
# Test with LoRA enabled (the digit 9 should be classified better)
test(test_dataloader=test_dataloader, model=base_model)

Testing: 100%|██████████| 1000/1000 [00:03<00:00, 276.29it/s]


Accuracy: 0.225

Wrong counts for the digit 0: 957
Wrong counts for the digit 1: 1125
Wrong counts for the digit 2: 825
Wrong counts for the digit 3: 891
Wrong counts for the digit 4: 926
Wrong counts for the digit 5: 727
Wrong counts for the digit 6: 370
Wrong counts for the digit 7: 1007
Wrong counts for the digit 8: 922
Wrong counts for the digit 9: 0


In [128]:
# Test with LoRA disabled
enable_disable_lora(enabled=False)
test(test_dataloader=test_dataloader, model=base_model)

Testing: 100%|██████████| 1000/1000 [00:03<00:00, 326.74it/s]


Accuracy: 0.965

Wrong counts for the digit 0: 24
Wrong counts for the digit 1: 15
Wrong counts for the digit 2: 27
Wrong counts for the digit 3: 31
Wrong counts for the digit 4: 36
Wrong counts for the digit 5: 37
Wrong counts for the digit 6: 34
Wrong counts for the digit 7: 46
Wrong counts for the digit 8: 40
Wrong counts for the digit 9: 63
